# Mini Data analytics with JSONPlaceholder









In [3]:
import pandas as pd
import requests
import matplotlib.pyplot as plt


**Fetching json data:**

In [4]:
users = requests.get("https://jsonplaceholder.typicode.com/users").json()
posts = requests.get("https://jsonplaceholder.typicode.com/posts").json()
todos = requests.get("https://jsonplaceholder.typicode.com/todos").json()
comments = requests.get("https://jsonplaceholder.typicode.com/comments").json()


**Creating DataFrames**

In [5]:
#Converting the (JSON) lists into DataFrames
users_df = pd.DataFrame(users)
posts_df = pd.DataFrame(posts)
todos_df = pd.DataFrame(todos)
comments_df = pd.DataFrame(comments)
#Some columns in the users df had nested dictionaries, by doing this they become separate columns
users_df = pd.json_normalize(users)

**Merging Dataframes**

In [6]:
users_posts_merged = users_df.merge(posts_df,how='left', left_on='id', right_on='userId', suffixes=['_user', '_post'])
posts_and_comments = posts_df.merge(comments_df, how='left', left_on='id', right_on='postId', suffixes=['_posts', "_comments"])
users_todos_merged = users_df.merge(todos_df, how='left', left_on='id', right_on='userId', suffixes=['_user', "_todo"])

**Ex 1. Number of posts per user:**

In [7]:
#Group by name and count the id_post
posts_per_user = users_posts_merged.groupby('name')['id_post'].count()
posts_per_user

,id_post
name,
Chelsey Dietrich,10
Clementina DuBuque,10
Clementine Bauch,10
Ervin Howell,10
Glenna Reichert,10
Kurtis Weissnat,10
Leanne Graham,10
Mrs. Dennis Schulist,10
Nicholas Runolfsdottir V,10


**Ex 2. Average comment per post**

In [12]:
#Group all comments by the post they belong and then .count() counts all the comments per post
comments_per_posts = posts_and_comments.groupby('postId').size()
avg_comments = comments_per_posts.mean()
comments_per_posts

,0
postId,
1,5
2,5
3,5
4,5
5,5
...,...
96,5
97,5
98,5


**Ex 3. Percentage of TODOs completed by users**

In [9]:
# Filter only completed TODOs
# Then group by user name and count how many tasks each user has completed
completed_todos =  users_todos_merged[users_todos_merged['completed'] == True] \
                                          .groupby('name')['completed'].count()
# Count the total TODOs per user
total_todos_per_user = users_todos_merged.groupby('name')['id_todo'].count()
# Then get the percentage of the completed TODOs
percentage_todos_completed = (completed_todos / total_todos_per_user) * 100
percentage_todos_completed

,0
name,
Chelsey Dietrich,60.0
Clementina DuBuque,60.0
Clementine Bauch,35.0
Ervin Howell,40.0
Glenna Reichert,40.0
Kurtis Weissnat,45.0
Leanne Graham,55.0
Mrs. Dennis Schulist,30.0
Nicholas Runolfsdottir V,55.0


**Ex 4. Top 5 most commented posts:**

In JSONPlaceholder, every post has exactly 5 comments, so top 5 most commented posts are arbitrary but still shown as required.

In [13]:
# Sort posts by number of comments and then printing the top 5 using .head(5)
top_5 = comments_per_posts.sort_values(ascending=False).head(5)
top_5

,0
postId,
1,5
2,5
3,5
4,5
5,5


## Visualizations

**1. Bar chart - User activity**

In [ ]:
# Create a bar chart showing the number of posts per user
posts_per_user.plot(kind='bar', color='skyblue')
plt.title('User activity')
plt.ylabel('Number of posts per user')
plt.xlabel('User name')
# Rotates the names for readability
plt.xticks(rotation=45)
# Add horizontal grid lines for easier visualization
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

**2. Pie chart - Percentage of Completed TODOs per User**

In [ ]:
# Creates a pie chart showing the percentage of completed TODOs per user, autopct is used to show the porcentages on the slices, and startangle is used for a better layout

percentage_todos_completed.plot(kind='pie', autopct='%1.1f%%', startangle=90, shadow= True)
plt.title('Percentage of tasks completed')
plt.ylabel('')
plt.show()

**3. Bar chart - Top 5 Most Commented Posts Bar Chart**

In [ ]:
# Create a bar chart showing the top 5 posts with the most comments
top_5.plot(kind='bar', color='green')
plt.title('Top 5 most commented posts')
plt.ylabel('Number comments')
plt.xlabel('Post ID')
# Keep x-axis labels horizontal for readability
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()